# PODCAST Benchmark — All Conditions Result Matrix

Loads results across **all conditions** (supersubject, subject_full, sig10, persubject_concat)
for foundation models and baselines. Generates CSVs under `results_allcond/`.

**Filters:** lag=0 only, epochs=100 for FM validation

In [91]:
import os
import re
import yaml
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

REPO_ROOT = Path(os.path.abspath('')).parent

# ── Directories ──
FM_RESULTS_DIR = REPO_ROOT / 'results' / 'foundation_models'
FM_SIG10_DIR = REPO_ROOT / 'results_sig10afterfix_and_superbeforefix' / 'foundation_models'
BASELINE_SUPER_DIR = REPO_ROOT / 'baseline-results'
BASELINE_SIG10_DIR = REPO_ROOT / 'results' / 'baseline'

# ── Task mapping ──
TASK_DIR_TO_NAME = {
    'content_noncontent': 'content_noncontent_task',
    'sentence_onset': 'sentence_onset_task',
    'iu_boundary': 'iu_boundary_task',
    'gpt_surprise': 'gpt_surprise_task',
    'gpt_surprise_multiclass': 'gpt_surprise_multiclass_task',
    'pos': 'pos_task',
    'word_embedding': 'word_embedding_decoding_task',
    'whisper_embedding': 'whisper_embedding_decoding_task',
    'volume_level': 'volume_level_decoding_task',
    'llm_decoding': 'llm_decoding_task',
    'llm_embedding_pretraining': 'llm_embedding_pretraining_task',
}

PRIMARY_METRIC = {
    'content_noncontent_task': 'roc_auc',
    'sentence_onset_task': 'roc_auc',
    'iu_boundary_task': 'roc_auc',
    'gpt_surprise_task': 'corr',
    'gpt_surprise_multiclass_task': 'roc_auc_multiclass',
    'pos_task': 'roc_auc_multiclass',
    'word_embedding_decoding_task': 'cosine_sim',
    'whisper_embedding_decoding_task': 'pairwise_accuracy',
    'volume_level_decoding_task': 'corr',
    'llm_decoding_task': 'perplexity_llm',
    'llm_embedding_pretraining_task': 'mse',
}

LOWER_IS_BETTER = {'perplexity_llm', 'mse', 'cross_entropy', 'bce_with_logits', 'cosine_dist', 'nll_embedding'}

FOUNDATION_MODELS = ['brainbert', 'diver', 'popt']
SUBJECTS = list(range(1, 10))  # 1..9
ALL_TASKS = sorted(PRIMARY_METRIC.keys())
TASK_SHORT = {t: t.replace('_task', '').replace('_decoding', '') for t in ALL_TASKS}

# ── Baseline prefix -> (display_name, task_name, n_folds) ──
# gpt_surprise is ambiguous (multiclass vs regression) — resolved via grep on config.yml
BASELINE_PREFIX_MAP = {
    'content_noncontent_task_sig_elecs_mlp_early_stop_roc': ('baseline', 'content_noncontent_task', 10),
    'iu_boundary_lr': ('baseline', 'iu_boundary_task', 5),
    'sentence_onset_lr': ('baseline', 'sentence_onset_task', 5),
    'pos_task_sig_elecs_without_other_classes': ('baseline', 'pos_task', 5),
    'ensemble_model_10_arbitrary': ('baseline_word_emb_arbitrary', 'word_embedding_decoding_task', 5),
    'ensemble_model_10_glove': ('baseline_word_emb_glove', 'word_embedding_decoding_task', 5),
    'ensemble_model_10_gpt2': ('baseline_word_emb_gpt2', 'word_embedding_decoding_task', 5),
    'neural_conv_whisper_embedding': ('baseline', 'whisper_embedding_decoding_task', 5),
    'volume_level_simple': ('baseline_volume_simple', 'volume_level_decoding_task', 5),
    'volume_level_torch_ridge': ('baseline_volume_torch_ridge', 'volume_level_decoding_task', 10),
    'llm_decoding_control': ('baseline_llm_control', 'llm_decoding_task', 5),
    'llm_token_finetune': ('baseline_llm_finetune', 'llm_decoding_task', 5),
}

# All unique baseline display names (rows in the matrix)
BASELINE_MODELS = sorted(set(v[0] for v in BASELINE_PREFIX_MAP.values()))

OUTPUT_DIR = Path(os.path.abspath('')) / 'results_allcond'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repo root:  {REPO_ROOT}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'FM models:  {FOUNDATION_MODELS}')
print(f'Baselines:  {BASELINE_MODELS}')

Repo root:  /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark
Output dir: /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/notebooks/results_allcond
FM models:  ['brainbert', 'diver', 'popt']
Baselines:  ['baseline', 'baseline_llm_control', 'baseline_llm_finetune', 'baseline_volume_simple', 'baseline_volume_torch_ridge', 'baseline_word_emb_arbitrary', 'baseline_word_emb_glove', 'baseline_word_emb_gpt2']


## Helper Functions

In [92]:
def is_valid_run(run_dir, required_epochs=100):
    """Check that a FM run dir has config.yml with correct epochs."""
    cfg_path = Path(run_dir) / 'config.yml'
    if not cfg_path.exists():
        return False
    try:
        with open(cfg_path) as f:
            cfg = yaml.safe_load(f)
        return cfg.get('training_params', {}).get('epochs') == required_epochs
    except Exception:
        return False


def find_latest_valid_run(condition_dir):
    """Find the most recent valid run dir (has config.yml w/ epochs=100 AND lag_performance.csv)."""
    if not condition_dir.exists():
        return None
    run_dirs = [p for p in condition_dir.iterdir() if p.is_dir()]
    valid = []
    for d in run_dirs:
        if not is_valid_run(d):
            continue
        if not (d / 'lag_performance.csv').exists():
            continue
        valid.append((os.path.getmtime(d), d))
    if not valid:
        return None
    valid.sort(reverse=True)
    return valid[0][1]


def read_lag0_metric(csv_path, task_name):
    """Read lag_performance.csv, filter to lag=0, return (mean, std) of test primary metric."""
    df = pd.read_csv(csv_path)
    lag0 = df[df['lags'] == 0]
    if lag0.empty:
        return np.nan, np.nan
    primary = PRIMARY_METRIC.get(task_name)
    if primary is None:
        return np.nan, np.nan
    mean_col = f'test_{primary}_mean'
    std_col = f'test_{primary}_std'
    mean_val = lag0.iloc[0][mean_col] if mean_col in lag0.columns else np.nan
    std_val = lag0.iloc[0][std_col] if std_col in lag0.columns else np.nan
    return mean_val, std_val


def strip_timestamp(dirname):
    """Remove trailing _YYYY-MM-DD-HH-MM-SS from a directory name."""
    return re.sub(r'_\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}$', '', dirname)


def grep_task_name(config_path):
    """Extract task_name from config.yml via text grep (avoids yaml.safe_load failures
    on configs containing numpy-serialized objects).
    Matches both top-level ('task_name: ...') and indented ('  task_name: ...') lines.
    """
    try:
        with open(config_path) as f:
            for line in f:
                stripped = line.strip()
                if stripped.startswith('task_name:'):
                    return stripped.split(':', 1)[1].strip()
    except Exception:
        pass
    return None


def resolve_baseline_info(dirpath):
    """Given a baseline run directory, return (display_name, task_name, n_folds) or None.
    Handles gpt_surprise ambiguity by grepping task_name from config.yml.
    """
    prefix = strip_timestamp(dirpath.name)
    if prefix == 'gpt_surprise':
        task_name = grep_task_name(dirpath / 'config.yml')
        if task_name and 'multiclass' in task_name:
            return ('baseline', 'gpt_surprise_multiclass_task', 10)
        elif task_name:
            return ('baseline', 'gpt_surprise_task', 10)
        return None
    return BASELINE_PREFIX_MAP.get(prefix)


print('Helper functions defined.')

Helper functions defined.


## Load FM Results (All Conditions)

In [93]:
# Storage: fm[condition][model][task] = (mean, std)   (for supersubject, persubject_concat)
#          fm[condition][subj][model][task] = (mean, std) (for subject_full, sig10)
fm = {
    'supersubject': defaultdict(dict),
    'persubject_concat': defaultdict(dict),
    'subject_full': {s: defaultdict(dict) for s in SUBJECTS},
    'sig10': {s: defaultdict(dict) for s in SUBJECTS},
}

missing_runs = []  # track dirs with config.yml but no lag_performance.csv

for model in FOUNDATION_MODELS:
    for task_dir, task_name in TASK_DIR_TO_NAME.items():

        # ── supersubject ──
        cond_dir = FM_RESULTS_DIR / model / task_dir / 'supersubject'
        run = find_latest_valid_run(cond_dir)
        if run:
            fm['supersubject'][model][task_name] = read_lag0_metric(run / 'lag_performance.csv', task_name)
        elif cond_dir.exists():
            missing_runs.append(str(cond_dir))

        # ── persubject_concat ──
        cond_dir = FM_RESULTS_DIR / model / task_dir / 'persubject_concat'
        run = find_latest_valid_run(cond_dir)
        if run:
            fm['persubject_concat'][model][task_name] = read_lag0_metric(run / 'lag_performance.csv', task_name)
        elif cond_dir.exists():
            missing_runs.append(str(cond_dir))

        # ── subject_full (per subject) ──
        # Path: results/foundation_models/{model}/{task}/subject_full/subject{N}_full/{run}
        for subj in SUBJECTS:
            cond_dir = FM_RESULTS_DIR / model / task_dir / 'subject_full' / f'subject{subj}_full'
            run = find_latest_valid_run(cond_dir)
            if run:
                fm['subject_full'][subj][model][task_name] = read_lag0_metric(run / 'lag_performance.csv', task_name)
            elif cond_dir.exists():
                missing_runs.append(str(cond_dir))

        # ── sig10 (per subject) ──
        # Path: results_sig10.../foundation_models/{model}/{task}/subject{N}_full/{run}
        for subj in SUBJECTS:
            cond_dir = FM_SIG10_DIR / model / task_dir / f'subject{subj}_full'
            run = find_latest_valid_run(cond_dir)
            if run:
                fm['sig10'][subj][model][task_name] = read_lag0_metric(run / 'lag_performance.csv', task_name)
            elif cond_dir.exists():
                missing_runs.append(str(cond_dir))

# Summary
for cond in ['supersubject', 'persubject_concat']:
    counts = {m: len(fm[cond].get(m, {})) for m in FOUNDATION_MODELS}
    print(f'FM {cond}: {counts}')
for cond in ['subject_full', 'sig10']:
    for subj in SUBJECTS:
        counts = {m: len(fm[cond][subj].get(m, {})) for m in FOUNDATION_MODELS}
        print(f'FM {cond} sub{subj}: {counts}')

if missing_runs:
    print(f'\nRuns with config.yml but NO lag_performance.csv ({len(missing_runs)}):')
    for path in missing_runs:
        print(f'  {path}')

FM supersubject: {'brainbert': 8, 'diver': 9, 'popt': 9}
FM persubject_concat: {'brainbert': 9, 'diver': 8, 'popt': 9}
FM subject_full sub1: {'brainbert': 8, 'diver': 8, 'popt': 9}
FM subject_full sub2: {'brainbert': 8, 'diver': 8, 'popt': 9}
FM subject_full sub3: {'brainbert': 8, 'diver': 8, 'popt': 9}
FM subject_full sub4: {'brainbert': 8, 'diver': 8, 'popt': 9}
FM subject_full sub5: {'brainbert': 8, 'diver': 8, 'popt': 9}
FM subject_full sub6: {'brainbert': 8, 'diver': 8, 'popt': 9}
FM subject_full sub7: {'brainbert': 8, 'diver': 8, 'popt': 9}
FM subject_full sub8: {'brainbert': 8, 'diver': 8, 'popt': 9}
FM subject_full sub9: {'brainbert': 8, 'diver': 7, 'popt': 9}
FM sig10 sub1: {'brainbert': 9, 'diver': 8, 'popt': 8}
FM sig10 sub2: {'brainbert': 9, 'diver': 8, 'popt': 8}
FM sig10 sub3: {'brainbert': 8, 'diver': 8, 'popt': 8}
FM sig10 sub4: {'brainbert': 9, 'diver': 8, 'popt': 8}
FM sig10 sub5: {'brainbert': 8, 'diver': 8, 'popt': 8}
FM sig10 sub6: {'brainbert': 9, 'diver': 8, 'pop

## Load Baseline Results

In [94]:
# baseline_super[display_name][task] = (mean, std)
# baseline_sig10[subj][display_name][task] = (mean, std)
baseline_super = defaultdict(dict)
baseline_sig10 = {s: defaultdict(dict) for s in SUBJECTS}
baseline_meta = {}  # (display_name, task) -> n_folds

# ── Supersubject baselines ──
if BASELINE_SUPER_DIR.exists():
    for d in sorted(BASELINE_SUPER_DIR.iterdir()):
        if not d.is_dir():
            continue
        csv_path = d / 'lag_performance.csv'
        if not csv_path.exists():
            continue
        info = resolve_baseline_info(d)
        if info is None:
            print(f'  [SKIP] unrecognized baseline: {d.name}')
            continue
        display_name, task_name, n_folds = info
        val = read_lag0_metric(csv_path, task_name)
        baseline_super[display_name][task_name] = val
        baseline_meta[(display_name, task_name)] = n_folds

print('Supersubject baselines loaded:')
for name in sorted(baseline_super):
    tasks = list(baseline_super[name].keys())
    print(f'  {name}: {[TASK_SHORT.get(t, t) for t in tasks]}')

# ── Sig10 baselines (per subject) ──
if BASELINE_SIG10_DIR.exists():
    for subj in SUBJECTS:
        sub_dir = BASELINE_SIG10_DIR / f'sub{subj}'
        if not sub_dir.exists():
            continue
        for d in sorted(sub_dir.iterdir()):
            if not d.is_dir():
                continue
            csv_path = d / 'lag_performance.csv'
            if not csv_path.exists():
                continue
            info = resolve_baseline_info(d)
            if info is None:
                print(f'  [SKIP] sig10 sub{subj} unrecognized: {d.name}')
                continue
            display_name, task_name, n_folds = info
            val = read_lag0_metric(csv_path, task_name)
            baseline_sig10[subj][display_name][task_name] = val
            baseline_meta[(display_name, task_name)] = n_folds

print('\nSig10 baselines loaded:')
for subj in SUBJECTS:
    names = sorted(baseline_sig10[subj].keys())
    n_tasks = sum(len(baseline_sig10[subj][n]) for n in names)
    print(f'  sub{subj}: {n_tasks} entries across {len(names)} baseline models')

Supersubject baselines loaded:
  baseline: ['content_noncontent', 'gpt_surprise_multiclass', 'gpt_surprise', 'iu_boundary', 'whisper_embedding', 'pos', 'sentence_onset']
  baseline_llm_control: ['llm']
  baseline_llm_finetune: ['llm']
  baseline_volume_simple: ['volume_level']
  baseline_volume_torch_ridge: ['volume_level']
  baseline_word_emb_arbitrary: ['word_embedding']
  baseline_word_emb_glove: ['word_embedding']
  baseline_word_emb_gpt2: ['word_embedding']

Sig10 baselines loaded:
  sub1: 7 entries across 2 baseline models
  sub2: 7 entries across 2 baseline models
  sub3: 7 entries across 2 baseline models
  sub4: 7 entries across 2 baseline models
  sub5: 7 entries across 2 baseline models
  sub6: 7 entries across 2 baseline models
  sub7: 7 entries across 2 baseline models
  sub8: 7 entries across 2 baseline models
  sub9: 7 entries across 2 baseline models


In [95]:
# ── Explore baseline-results/ directory ──
print('Contents of baseline-results/:')
print(f'  {"Directory":<65s} {"config.yml":>10s} {"lag_perf.csv":>13s} {"task_name (from config)"}')
print('-' * 130)
for d in sorted(BASELINE_SUPER_DIR.iterdir()):
    if not d.is_dir():
        continue
    has_cfg = (d / 'config.yml').exists()
    has_csv = (d / 'lag_performance.csv').exists()
    task = grep_task_name(d / 'config.yml') if has_cfg else '—'
    info = resolve_baseline_info(d)
    mapped = f'→ {info[0]} | {TASK_SHORT.get(info[1], info[1])} | {info[2]}folds' if info else '→ [UNMAPPED]'
    print(f'  {d.name:<65s} {"YES" if has_cfg else "NO":>10s} {"YES" if has_csv else "NO":>13s}   {task:<40s} {mapped}')

Contents of baseline-results/:
  Directory                                                         config.yml  lag_perf.csv task_name (from config)
----------------------------------------------------------------------------------------------------------------------------------
  content_noncontent_task_sig_elecs_mlp_early_stop_roc_2025-12-19-00-34-17        YES           YES   content_noncontent_task                  → baseline | content_noncontent | 10folds
  ensemble_model_10_arbitrary_2025-12-19-00-17-32                          YES           YES   word_embedding_decoding_task             → baseline_word_emb_arbitrary | word_embedding | 5folds
  ensemble_model_10_glove_2025-12-19-00-17-41                              YES           YES   word_embedding_decoding_task             → baseline_word_emb_glove | word_embedding | 5folds
  ensemble_model_10_gpt2_2026-02-17-19-54-55                               YES           YES   word_embedding_decoding_task             → baseline_word_emb_

In [96]:
# ── Baseline primary metric values at lag=0 ──
rows = []
for d in sorted(BASELINE_SUPER_DIR.iterdir()):
    if not d.is_dir():
        continue
    csv_path = d / 'lag_performance.csv'
    if not csv_path.exists():
        continue
    info = resolve_baseline_info(d)
    if info is None:
        continue
    display_name, task_name, n_folds = info
    primary = PRIMARY_METRIC.get(task_name, '?')
    col = f'test_{primary}_mean'
    df = pd.read_csv(csv_path)
    lag0 = df[df['lags'] == 0]
    val = lag0.iloc[0][col] if (not lag0.empty and col in lag0.columns) else np.nan
    # Also list all available test_*_mean columns
    test_cols = [c for c in df.columns if c.startswith('test_') and c.endswith('_mean')]
    rows.append({
        'directory': d.name,
        'display_name': display_name,
        'task': TASK_SHORT.get(task_name, task_name),
        'primary_metric': primary,
        'value_at_lag0': round(val, 4) if pd.notna(val) else 'NaN',
        'n_folds': n_folds,
        'available_test_metrics': ', '.join(c.replace('test_', '').replace('_mean', '') for c in test_cols),
    })

bl_explore_df = pd.DataFrame(rows)
print('=== Baseline primary metric at lag=0 ===')
display(bl_explore_df)

=== Baseline primary metric at lag=0 ===


,directory,display_name,task,primary_metric,value_at_lag0,n_folds,available_test_metrics
0,content_noncontent_task_sig_elecs_mlp_early_st...,baseline,content_noncontent,roc_auc,0.5738,10,"bce, roc_auc, f1"
1,ensemble_model_10_arbitrary_2025-12-19-00-17-32,baseline_word_emb_arbitrary,word_embedding,cosine_sim,0.0153,5,"mse, nll_embedding, cosine_sim, word_avg_auc_r..."
2,ensemble_model_10_glove_2025-12-19-00-17-41,baseline_word_emb_glove,word_embedding,cosine_sim,0.6161,5,"mse, nll_embedding, cosine_sim, word_avg_auc_r..."
3,ensemble_model_10_gpt2_2026-02-17-19-54-55,baseline_word_emb_gpt2,word_embedding,cosine_sim,0.6152,5,"mse, nll_embedding, cosine_sim, pairwise_accur..."
4,gpt_surprise_2025-12-19-00-18-43,baseline,gpt_surprise_multiclass,roc_auc_multiclass,0.5253,10,"cross_entropy, roc_auc_multiclass, f1"
5,gpt_surprise_2025-12-19-00-18-44,baseline,gpt_surprise,corr,0.0332,10,"mse, corr"
6,iu_boundary_lr_2026-02-26-09-46-13,baseline,iu_boundary,roc_auc,0.4831,5,"bce, roc_auc, sensitivity, specificity, f1, pr..."
7,llm_decoding_control_2025-12-28-15-55-38,baseline_llm_control,llm,perplexity_llm,NaN,5,"cross_entropy, perplexity"
8,llm_token_finetune_2025-12-26-12-44-36,baseline_llm_finetune,llm,perplexity_llm,NaN,5,"cross_entropy, perplexity"
9,neural_conv_whisper_embedding_2026-02-17-19-25-13,baseline,whisper_embedding,pairwise_accuracy,0.6854,5,"mse, cosine_sim, pairwise_accuracy"


## Build Result Matrices

In [97]:
# ── Supersubject: FM only ──
mat_super_mean = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=float)
mat_super_std = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=float)

for model in FOUNDATION_MODELS:
    for task in ALL_TASKS:
        m, s = fm['supersubject'].get(model, {}).get(task, (np.nan, np.nan))
        mat_super_mean.loc[model, task] = m
        mat_super_std.loc[model, task] = s

# Display with mean +/- fold_std
display_super = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=object)
for model in FOUNDATION_MODELS:
    for task in ALL_TASKS:
        m = mat_super_mean.loc[model, task]
        s = mat_super_std.loc[model, task]
        if pd.notna(m):
            if pd.notna(s):
                display_super.loc[model, task] = f'{m:.4f} +/- {s:.4f}'
            else:
                display_super.loc[model, task] = f'{m:.4f}'
        else:
            display_super.loc[model, task] = '-'

display_super.columns = [TASK_SHORT[c] for c in display_super.columns]
print('=== Supersubject — FM only (lag=0, test split) ===')
print('  mean +/- std across folds')
print()
display(display_super)

=== Supersubject — FM only (lag=0, test split) ===
  mean +/- std across folds



,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5954 +/- 0.0035,0.5221 +/- 0.0198,0.0388 +/- 0.0304,0.7192 +/- 0.0248,-,0.0688 +/- 0.0011,0.5663 +/- 0.0023,0.7233 +/- 0.0346,-,0.6939 +/- 0.0110,-
diver,0.5780 +/- 0.0076,0.5251 +/- 0.0001,0.0759 +/- 0.0116,0.6806 +/- 0.0258,-,0.0114 +/- 0.0001,0.5689 +/- 0.0151,0.7068 +/- 0.0201,-,0.6981 +/- 0.0123,0.0664 +/- 0.0024
popt,0.4769 +/- 0.0023,0.5241 +/- 0.0022,0.0111 +/- 0.0180,0.5481 +/- 0.0551,-,0.0113 +/- 0.0002,0.5124 +/- 0.0248,0.5235 +/- 0.0757,-,0.6133 +/- 0.0148,-0.0621 +/- 0.0054


In [98]:
# ── Subject_full: FM only, per subject + averaged ──
# Each value is (mean_across_folds, std_across_folds). For per-subject matrices, show mean +/- fold_std.
# For subject-averaged, show mean +/- subject_std across subjects (of the fold-mean values).

per_subj_full = {}
per_subj_full_std = {}  # fold std per subject
for subj in SUBJECTS:
    mat_m = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=float)
    mat_s = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=float)
    for model in FOUNDATION_MODELS:
        for task in ALL_TASKS:
            m, s = fm['subject_full'][subj].get(model, {}).get(task, (np.nan, np.nan))
            mat_m.loc[model, task] = m
            mat_s.loc[model, task] = s
    per_subj_full[subj] = mat_m
    per_subj_full_std[subj] = mat_s

stacked = np.stack([per_subj_full[s].values.astype(float) for s in SUBJECTS], axis=0)
mean_subj_full = pd.DataFrame(np.nanmean(stacked, axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)
std_subj_full = pd.DataFrame(np.nanstd(stacked, axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)
count_subj_full = pd.DataFrame(np.sum(~np.isnan(stacked), axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)

# Display mean +/- subject_std
display_sf = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=object)
for model in FOUNDATION_MODELS:
    for task in ALL_TASKS:
        m = mean_subj_full.loc[model, task]
        s = std_subj_full.loc[model, task]
        n = count_subj_full.loc[model, task]
        if pd.notna(m):
            display_sf.loc[model, task] = f'{m:.4f} +/- {s:.4f} (n={int(n)})'
        else:
            display_sf.loc[model, task] = '-'

display_sf.columns = [TASK_SHORT[c] for c in display_sf.columns]
print('=== Subject_full — Subject-averaged (lag=0, test split) ===')
print('  mean +/- std across subjects (each subject value is fold-averaged)')
print('  FM only (no baselines for this condition)')
print()
display(display_sf)

=== Subject_full — Subject-averaged (lag=0, test split) ===
  mean +/- std across subjects (each subject value is fold-averaged)
  FM only (no baselines for this condition)



/tmp/ipykernel_1817900/1484401115.py:19: RuntimeWarning: Mean of empty slice
  mean_subj_full = pd.DataFrame(np.nanmean(stacked, axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)
/global/u1/a/ahhyun/.conda/envs/DIVER/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5371 +/- 0.0229 (n=9),0.5044 +/- 0.0123 (n=9),0.0056 +/- 0.0227 (n=9),0.6040 +/- 0.0466 (n=9),-,0.0534 +/- 0.0179 (n=9),0.5232 +/- 0.0160 (n=9),0.5912 +/- 0.0602 (n=9),-,0.5605 +/- 0.0258 (n=9),-
diver,0.5509 +/- 0.0344 (n=8),0.5136 +/- 0.0173 (n=9),0.0408 +/- 0.0268 (n=9),0.6573 +/- 0.0724 (n=9),-,0.0112 +/- 0.0001 (n=9),0.5513 +/- 0.0323 (n=9),0.6551 +/- 0.0710 (n=9),-,0.6285 +/- 0.0524 (n=9),-
popt,0.5030 +/- 0.0235 (n=9),0.5020 +/- 0.0131 (n=9),0.0051 +/- 0.0116 (n=9),0.5126 +/- 0.0280 (n=9),-,0.0115 +/- 0.0001 (n=9),0.5068 +/- 0.0108 (n=9),0.5022 +/- 0.0287 (n=9),-,0.5009 +/- 0.0052 (n=9),-0.0869 +/- 0.0029 (n=9)


In [99]:
# ── Sig10: FM only, per subject + averaged ──
# NOTE: sig10 results use only 2 folds (fold_1, fold_5). _mean/_std are across these 2 folds.

per_subj_sig10 = {}
per_subj_sig10_std = {}
for subj in SUBJECTS:
    mat_m = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=float)
    mat_s = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=float)
    for model in FOUNDATION_MODELS:
        for task in ALL_TASKS:
            m, s = fm['sig10'][subj].get(model, {}).get(task, (np.nan, np.nan))
            mat_m.loc[model, task] = m
            mat_s.loc[model, task] = s
    per_subj_sig10[subj] = mat_m
    per_subj_sig10_std[subj] = mat_s

stacked_sig10 = np.stack([per_subj_sig10[s].values.astype(float) for s in SUBJECTS], axis=0)
mean_sig10 = pd.DataFrame(np.nanmean(stacked_sig10, axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)
std_sig10 = pd.DataFrame(np.nanstd(stacked_sig10, axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)
count_sig10 = pd.DataFrame(np.sum(~np.isnan(stacked_sig10), axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)

# Display
display_sig10 = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=object)
for model in FOUNDATION_MODELS:
    for task in ALL_TASKS:
        m = mean_sig10.loc[model, task]
        s = std_sig10.loc[model, task]
        n = count_sig10.loc[model, task]
        if pd.notna(m):
            display_sig10.loc[model, task] = f'{m:.4f} +/- {s:.4f} (n={int(n)})'
        else:
            display_sig10.loc[model, task] = '-'

display_sig10.columns = [TASK_SHORT[c] for c in display_sig10.columns]
print('=== Sig10 — FM only, Subject-averaged (lag=0, test split) ===')
print('  mean +/- std across subjects (each subject value is mean of fold_1 & fold_5)')
print()
display(display_sig10)

=== Sig10 — FM only, Subject-averaged (lag=0, test split) ===
  mean +/- std across subjects (each subject value is mean of fold_1 & fold_5)



/tmp/ipykernel_1817900/454325336.py:18: RuntimeWarning: Mean of empty slice
  mean_sig10 = pd.DataFrame(np.nanmean(stacked_sig10, axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)
/global/u1/a/ahhyun/.conda/envs/DIVER/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5504 +/- 0.0210 (n=9),0.5085 +/- 0.0128 (n=9),0.0016 +/- 0.0244 (n=9),0.6469 +/- 0.0528 (n=9),-,0.0184 +/- 0.0036 (n=9),0.5262 +/- 0.0242 (n=9),0.6072 +/- 0.0659 (n=9),0.1089 +/- 0.0286 (n=5),0.6037 +/- 0.0484 (n=9),-
diver,0.5747 +/- 0.0418 (n=9),0.5251 +/- 0.0217 (n=9),0.0527 +/- 0.0298 (n=9),0.6625 +/- 0.0658 (n=9),-,0.0108 +/- 0.0001 (n=9),0.5682 +/- 0.0427 (n=9),0.6443 +/- 0.0481 (n=9),-,0.6784 +/- 0.0586 (n=9),-
popt,0.5162 +/- 0.0239 (n=9),0.5024 +/- 0.0184 (n=9),0.0022 +/- 0.0193 (n=9),0.5583 +/- 0.0429 (n=9),-,0.0117 +/- 0.0001 (n=9),0.5204 +/- 0.0180 (n=9),0.5196 +/- 0.0302 (n=9),-,0.5414 +/- 0.0374 (n=9),-


In [100]:
# ── Persubject_concat: FM only ──
mat_psconcat_mean = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=float)
mat_psconcat_std = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=float)
for model in FOUNDATION_MODELS:
    for task in ALL_TASKS:
        m, s = fm['persubject_concat'].get(model, {}).get(task, (np.nan, np.nan))
        mat_psconcat_mean.loc[model, task] = m
        mat_psconcat_std.loc[model, task] = s

display_psc = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=object)
for model in FOUNDATION_MODELS:
    for task in ALL_TASKS:
        m = mat_psconcat_mean.loc[model, task]
        s = mat_psconcat_std.loc[model, task]
        if pd.notna(m):
            if pd.notna(s):
                display_psc.loc[model, task] = f'{m:.4f} +/- {s:.4f}'
            else:
                display_psc.loc[model, task] = f'{m:.4f}'
        else:
            display_psc.loc[model, task] = '-'

display_psc.columns = [TASK_SHORT[c] for c in display_psc.columns]
print('=== Persubject_concat (lag=0, test split) ===')
print('  mean +/- std across folds')
print('  FM only (no baselines for this condition)')
print()
display(display_psc)

=== Persubject_concat (lag=0, test split) ===
  mean +/- std across folds
  FM only (no baselines for this condition)



,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5766 +/- 0.0141,0.5241 +/- 0.0159,0.0374 +/- 0.0132,0.7222 +/- 0.0288,-,0.1026 +/- 0.0011,0.5593 +/- 0.0096,0.7124 +/- 0.0307,-,0.6779 +/- 0.0044,0.0425 +/- 0.0161
diver,0.6559 +/- 0.0154,0.5508 +/- 0.0179,0.0965 +/- 0.0009,0.7808 +/- 0.0218,-,0.0114 +/- 0.0002,0.6489 +/- 0.0080,0.7932 +/- 0.0020,-,0.7931 +/- 0.0010,-
popt,0.5251 +/- 0.0291,0.5208 +/- 0.0011,0.0368 +/- 0.0080,0.6984 +/- 0.0040,-,0.0132 +/- 0.0003,0.5226 +/- 0.0214,0.6830 +/- 0.0244,-,0.6244 +/- 0.0021,-0.0169 +/- 0.0121


## Baseline Results (separate from FM)

Baselines are kept in separate dataframes because they use different electrode/condition setups:
- `baseline-results/` — uses all channels (similar to subject_full), fold-averaged
- `results/baseline/sub{N}/` — sig10 channels, per-subject, fold_1 & fold_5

In [101]:
# ── Baseline: baseline-results/ (all channels, fold-averaged) ──
bl_super_names = sorted(baseline_super.keys())
mat_bl_super_mean = pd.DataFrame(index=bl_super_names, columns=ALL_TASKS, dtype=float)
mat_bl_super_std = pd.DataFrame(index=bl_super_names, columns=ALL_TASKS, dtype=float)

for bl_name in bl_super_names:
    for task in ALL_TASKS:
        m, s = baseline_super[bl_name].get(task, (np.nan, np.nan))
        mat_bl_super_mean.loc[bl_name, task] = m
        mat_bl_super_std.loc[bl_name, task] = s

display_bl_super = pd.DataFrame(index=bl_super_names, columns=ALL_TASKS, dtype=object)
for bl_name in bl_super_names:
    for task in ALL_TASKS:
        m = mat_bl_super_mean.loc[bl_name, task]
        s = mat_bl_super_std.loc[bl_name, task]
        nf = baseline_meta.get((bl_name, task))
        if pd.notna(m):
            fold_info = f' ({nf}f)' if nf else ''
            if pd.notna(s):
                display_bl_super.loc[bl_name, task] = f'{m:.4f} +/- {s:.4f}{fold_info}'
            else:
                display_bl_super.loc[bl_name, task] = f'{m:.4f}{fold_info}'
        else:
            display_bl_super.loc[bl_name, task] = '-'

display_bl_super.columns = [TASK_SHORT[c] for c in display_bl_super.columns]
print('=== Baseline — from baseline-results/ (all channels, fold-averaged) ===')
print('  mean +/- std across folds, (Nf) = number of folds')
print()
display(display_bl_super)

=== Baseline — from baseline-results/ (all channels, fold-averaged) ===
  mean +/- std across folds, (Nf) = number of folds



,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
baseline,0.5738 +/- 0.0177 (10f),0.5253 +/- 0.0197 (10f),0.0332 +/- 0.0348 (10f),0.4831 +/- 0.0702 (5f),-,-,0.5191 +/- 0.0137 (5f),0.8800 +/- 0.0435 (5f),-,0.6854 +/- 0.0010 (5f),-
baseline_llm_control,-,-,-,-,-,-,-,-,-,-,-
baseline_llm_finetune,-,-,-,-,-,-,-,-,-,-,-
baseline_volume_simple,-,-,-,-,-,-,-,-,0.3380 +/- 0.0472 (5f),-,-
baseline_volume_torch_ridge,-,-,-,-,-,-,-,-,0.3310 +/- 0.0712 (10f),-,-
baseline_word_emb_arbitrary,-,-,-,-,-,-,-,-,-,-,0.0153 +/- 0.0064 (5f)
baseline_word_emb_glove,-,-,-,-,-,-,-,-,-,-,0.6161 +/- 0.0135 (5f)
baseline_word_emb_gpt2,-,-,-,-,-,-,-,-,-,-,0.6152 +/- 0.0148 (5f)


In [ ]:
# ── Baseline: results/baseline/supersubject/ (new, fold-averaged) ──
BASELINE_SUPER_NEW_DIR = REPO_ROOT / 'results' / 'baseline' / 'supersubject'

baseline_super_new = defaultdict(dict)
baseline_meta_new = {}

if BASELINE_SUPER_NEW_DIR.exists():
    for d in sorted(BASELINE_SUPER_NEW_DIR.iterdir()):
        if not d.is_dir():
            continue
        csv_path = d / 'lag_performance.csv'
        if not csv_path.exists():
            continue
        info = resolve_baseline_info(d)
        if info is None:
            print(f'  [SKIP] unrecognized baseline: {d.name}')
            continue
        display_name, task_name, n_folds = info
        val = read_lag0_metric(csv_path, task_name)
        baseline_super_new[display_name][task_name] = val
        baseline_meta_new[(display_name, task_name)] = n_folds

print(f'Loaded from: {BASELINE_SUPER_NEW_DIR}')
for name in sorted(baseline_super_new):
    tasks = list(baseline_super_new[name].keys())
    print(f'  {name}: {[TASK_SHORT.get(t, t) for t in tasks]}')

# Build matrix
bl_super_new_names = sorted(baseline_super_new.keys())
mat_bl_super_new_mean = pd.DataFrame(index=bl_super_new_names, columns=ALL_TASKS, dtype=float)
mat_bl_super_new_std = pd.DataFrame(index=bl_super_new_names, columns=ALL_TASKS, dtype=float)

for bl_name in bl_super_new_names:
    for task in ALL_TASKS:
        m, s = baseline_super_new[bl_name].get(task, (np.nan, np.nan))
        mat_bl_super_new_mean.loc[bl_name, task] = m
        mat_bl_super_new_std.loc[bl_name, task] = s

display_bl_super_new = pd.DataFrame(index=bl_super_new_names, columns=ALL_TASKS, dtype=object)
for bl_name in bl_super_new_names:
    for task in ALL_TASKS:
        m = mat_bl_super_new_mean.loc[bl_name, task]
        s = mat_bl_super_new_std.loc[bl_name, task]
        nf = baseline_meta_new.get((bl_name, task))
        if pd.notna(m):
            fold_info = f' ({nf}f)' if nf else ''
            if pd.notna(s):
                display_bl_super_new.loc[bl_name, task] = f'{m:.4f} +/- {s:.4f}{fold_info}'
            else:
                display_bl_super_new.loc[bl_name, task] = f'{m:.4f}{fold_info}'
        else:
            display_bl_super_new.loc[bl_name, task] = '-'

display_bl_super_new.columns = [TASK_SHORT[c] for c in display_bl_super_new.columns]
print()
print('=== Baseline — from results/baseline/supersubject/ (fold-averaged) ===')
print('  mean +/- std across folds, (Nf) = number of folds')
print()
display(display_bl_super_new)


In [102]:
# ── Baseline: sig10, per-subject + averaged ──
bl_sig10_names = sorted(set(
    name for subj in SUBJECTS for name in baseline_sig10[subj].keys()
))

per_subj_bl_sig10 = {}
per_subj_bl_sig10_std = {}
for subj in SUBJECTS:
    mat_m = pd.DataFrame(index=bl_sig10_names, columns=ALL_TASKS, dtype=float)
    mat_s = pd.DataFrame(index=bl_sig10_names, columns=ALL_TASKS, dtype=float)
    for bl_name in bl_sig10_names:
        for task in ALL_TASKS:
            m, s = baseline_sig10[subj].get(bl_name, {}).get(task, (np.nan, np.nan))
            mat_m.loc[bl_name, task] = m
            mat_s.loc[bl_name, task] = s
    per_subj_bl_sig10[subj] = mat_m
    per_subj_bl_sig10_std[subj] = mat_s

if bl_sig10_names:
    stacked_bl_sig10 = np.stack([per_subj_bl_sig10[s].values.astype(float) for s in SUBJECTS], axis=0)
    mean_bl_sig10 = pd.DataFrame(np.nanmean(stacked_bl_sig10, axis=0), index=bl_sig10_names, columns=ALL_TASKS)
    std_bl_sig10 = pd.DataFrame(np.nanstd(stacked_bl_sig10, axis=0), index=bl_sig10_names, columns=ALL_TASKS)
    count_bl_sig10 = pd.DataFrame(np.sum(~np.isnan(stacked_bl_sig10), axis=0), index=bl_sig10_names, columns=ALL_TASKS)

    display_bl_sig10 = pd.DataFrame(index=bl_sig10_names, columns=ALL_TASKS, dtype=object)
    for bl_name in bl_sig10_names:
        for task in ALL_TASKS:
            m = mean_bl_sig10.loc[bl_name, task]
            s = std_bl_sig10.loc[bl_name, task]
            n = count_bl_sig10.loc[bl_name, task]
            if pd.notna(m):
                display_bl_sig10.loc[bl_name, task] = f'{m:.4f} +/- {s:.4f} (n={int(n)})'
            else:
                display_bl_sig10.loc[bl_name, task] = '-'

    display_bl_sig10.columns = [TASK_SHORT[c] for c in display_bl_sig10.columns]
    print('=== Baseline — sig10, Subject-averaged (lag=0, test split) ===')
    print('  mean +/- std across subjects (each subject value is mean of fold_1 & fold_5)')
    print()
    display(display_bl_sig10)
else:
    print('No sig10 baselines found.')

=== Baseline — sig10, Subject-averaged (lag=0, test split) ===
  mean +/- std across subjects (each subject value is mean of fold_1 & fold_5)



/tmp/ipykernel_1817900/3059233847.py:21: RuntimeWarning: Mean of empty slice
  mean_bl_sig10 = pd.DataFrame(np.nanmean(stacked_bl_sig10, axis=0), index=bl_sig10_names, columns=ALL_TASKS)
/global/u1/a/ahhyun/.conda/envs/DIVER/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
baseline,0.5894 +/- 0.0354 (n=9),0.5221 +/- 0.0184 (n=9),0.0632 +/- 0.0505 (n=9),0.7958 +/- 0.0417 (n=9),-,-,-,0.8431 +/- 0.0460 (n=9),-,0.6681 +/- 0.0603 (n=9),-
baseline_word_emb_glove,-,-,-,-,-,-,-,-,-,-,0.6274 +/- 0.0058 (n=9)


In [103]:
# ── Per-subject detail: subject_full (mean +/- fold_std) ──
for subj in SUBJECTS:
    display_m = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=object)
    for model in FOUNDATION_MODELS:
        for task in ALL_TASKS:
            m = per_subj_full[subj].loc[model, task]
            s = per_subj_full_std[subj].loc[model, task]
            if pd.notna(m):
                if pd.notna(s):
                    display_m.loc[model, task] = f'{m:.4f} +/- {s:.4f}'
                else:
                    display_m.loc[model, task] = f'{m:.4f}'
            else:
                display_m.loc[model, task] = '-'
    display_m.columns = [TASK_SHORT[c] for c in display_m.columns]
    print(f'\n=== Subject_full — Subject {subj} (lag=0, test, mean +/- fold_std) ===')
    display(display_m)


=== Subject_full — Subject 1 (lag=0, test, mean +/- fold_std) ===


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5036 +/- 0.0149,0.4915 +/- 0.0002,-0.0087 +/- 0.0155,0.5883 +/- 0.0347,-,0.0399 +/- 0.0004,0.5017 +/- 0.0026,0.6070 +/- 0.0366,-,0.5743 +/- 0.0029,-
diver,0.5637 +/- 0.0002,0.5227 +/- 0.0050,0.0422 +/- 0.0001,0.6771 +/- 0.0243,-,0.0110 +/- 0.0002,0.5741 +/- 0.0043,0.6240 +/- 0.0146,-,0.6715 +/- 0.0075,-
popt,0.5321 +/- 0.0169,0.4998 +/- 0.0003,-0.0093 +/- 0.0046,0.5481 +/- 0.0144,-,0.0114 +/- 0.0002,0.5163 +/- 0.0013,0.5034 +/- 0.0125,-,0.5011 +/- 0.0028,-0.0891 +/- 0.0374



=== Subject_full — Subject 2 (lag=0, test, mean +/- fold_std) ===


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5293 +/- 0.0018,0.5013 +/- 0.0024,-0.0236 +/- 0.0128,0.5744 +/- 0.0079,-,0.0356 +/- 0.0007,0.5218 +/- 0.0024,0.5317 +/- 0.0376,-,0.5280 +/- 0.0021,-
diver,0.5303 +/- 0.0293,0.4926 +/- 0.0072,0.0070 +/- 0.0046,0.5794 +/- 0.0437,-,0.0110 +/- 0.0002,0.5233 +/- 0.0137,0.5909 +/- 0.0537,-,0.5751 +/- 0.0206,-
popt,0.5274 +/- 0.0103,0.4869 +/- 0.0088,-0.0078 +/- 0.0252,0.4737 +/- 0.0203,-,0.0116 +/- 0.0002,0.5075 +/- 0.0133,0.4579 +/- 0.0342,-,0.5026 +/- 0.0036,-0.0846 +/- 0.0397



=== Subject_full — Subject 3 (lag=0, test, mean +/- fold_std) ===


...

,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5494 +/- 0.0055,0.5109 +/- 0.0015,0.0124 +/- 0.0234,0.6592 +/- 0.0015,-,0.0431 +/- 0.0004,0.5413 +/- 0.0123,0.6750 +/- 0.0304,-,0.5817 +/- 0.0091,-
diver,0.5954 +/- 0.0197,0.5172 +/- 0.0094,0.0913 +/- 0.0218,0.5868 +/- 0.0164,-,0.0112 +/- 0.0002,0.5723 +/- 0.0020,0.6175 +/- 0.0371,-,0.6321 +/- 0.0183,-
popt,0.4879 +/- 0.0088,0.5263 +/- 0.0057,0.0115 +/- 0.0031,0.5650 +/- 0.0174,-,0.0115 +/- 0.0001,0.4862 +/- 0.0056,0.5584 +/- 0.0088,-,0.5054 +/- 0.0021,-0.0866 +/- 0.0379



=== Subject_full — Subject 8 (lag=0, test, mean +/- fold_std) ===


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5392 +/- 0.0048,0.5131 +/- 0.0193,-0.0098 +/- 0.0280,0.6701 +/- 0.0412,-,0.0309 +/- 0.0004,0.5196 +/- 0.0018,0.6621 +/- 0.0185,-,0.5780 +/- 0.0008,-
diver,0.4878 +/- 0.0033,0.5004 +/- 0.0038,0.0138 +/- 0.0172,0.5164 +/- 0.0263,-,0.0111 +/- 0.0001,0.4784 +/- 0.0004,0.5085 +/- 0.0336,-,0.5105 +/- 0.0063,-
popt,0.5187 +/- 0.0104,0.4953 +/- 0.0199,-0.0078 +/- 0.0085,0.5064 +/- 0.0174,-,0.0116 +/- 0.0003,0.5200 +/- 0.0034,0.5378 +/- 0.0157,-,0.4956 +/- 0.0041,-0.0866 +/- 0.0388



=== Subject_full — Subject 9 (lag=0, test, mean +/- fold_std) ===


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5271 +/- 0.0079,0.5233 +/- 0.0281,0.0298 +/- 0.0133,0.5451 +/- 0.0570,-,0.0723 +/- 0.0001,0.5248 +/- 0.0259,0.5695 +/- 0.0259,-,0.5538 +/- 0.0019,-
diver,-,0.5401 +/- 0.0023,0.0695 +/- 0.0087,0.7252 +/- 0.0268,-,0.0113 +/- 0.0002,0.5793 +/- 0.0079,0.7450 +/- 0.0131,-,0.6454 +/- 0.0021,-
popt,0.4932 +/- 0.0078,0.5226 +/- 0.0048,0.0084 +/- 0.0100,0.4836 +/- 0.0402,-,0.0113 +/- 0.0001,0.5004 +/- 0.0037,0.4849 +/- 0.0300,-,0.5065 +/- 0.0062,-0.0838 +/- 0.0334


In [104]:
# ── Per-subject detail: sig10 FM only (mean +/- fold_std, 2 folds) ──
for subj in SUBJECTS:
    display_m = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=object)
    for model in FOUNDATION_MODELS:
        for task in ALL_TASKS:
            m = per_subj_sig10[subj].loc[model, task]
            s = per_subj_sig10_std[subj].loc[model, task]
            if pd.notna(m):
                if pd.notna(s):
                    display_m.loc[model, task] = f'{m:.4f} +/- {s:.4f}'
                else:
                    display_m.loc[model, task] = f'{m:.4f}'
            else:
                display_m.loc[model, task] = '-'
    display_m.columns = [TASK_SHORT[c] for c in display_m.columns]
    print(f'\n=== Sig10 FM — Subject {subj} (lag=0, test, mean +/- fold_std) ===')
    display(display_m)


=== Sig10 FM — Subject 1 (lag=0, test, mean +/- fold_std) ===


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5509 +/- 0.0139,0.4946 +/- 0.0001,-0.0208 +/- 0.0105,0.6443 +/- 0.0124,-,0.0151 +/- 0.0000,0.4973 +/- 0.0005,0.6615 +/- 0.0220,0.0834 +/- 0.0254,0.6171 +/- 0.0175,-
diver,0.5708 +/- 0.0020,0.5290 +/- 0.0196,0.0770 +/- 0.0256,0.6736 +/- 0.0179,-,0.0107 +/- 0.0002,0.5707 +/- 0.0128,0.6335 +/- 0.0522,-,0.6999 +/- 0.0242,-
popt,0.4894 +/- 0.0132,0.5127 +/- 0.0112,0.0011 +/- 0.0016,0.5402 +/- 0.0094,-,0.0117 +/- 0.0002,0.5053 +/- 0.0016,0.5094 +/- 0.0114,-,0.5070 +/- 0.0066,-



=== Sig10 FM — Subject 2 (lag=0, test, mean +/- fold_std) ===


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5203 +/- 0.0084,0.5090 +/- 0.0050,-0.0225 +/- 0.0197,0.6052 +/- 0.0139,-,0.0183 +/- 0.0001,0.4921 +/- 0.0097,0.5405 +/- 0.0087,0.0821 +/- 0.0259,0.5404 +/- 0.0041,-
diver,0.5195 +/- 0.0308,0.5094 +/- 0.0075,0.0280 +/- 0.0169,0.5650 +/- 0.0084,-,0.0108 +/- 0.0002,0.5125 +/- 0.0189,0.5808 +/- 0.0889,-,0.5905 +/- 0.0097,-
popt,0.5122 +/- 0.0003,0.4637 +/- 0.0253,-0.0084 +/- 0.0018,0.5362 +/- 0.0094,-,0.0117 +/- 0.0002,0.5014 +/- 0.0080,0.4600 +/- 0.0152,-,0.5029 +/- 0.0075,-



=== Sig10 FM — Subject 3 (lag=0, test, mean +/- fold_std) ===


...

,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5640 +/- 0.0157,0.5204 +/- 0.0013,0.0385 +/- 0.0087,0.6840 +/- 0.0104,-,0.0148 +/- 0.0000,0.5351 +/- 0.0054,0.6848 +/- 0.0251,-,0.6467 +/- 0.0005,-
diver,0.6096 +/- 0.0015,0.5225 +/- 0.0194,0.0634 +/- 0.0688,0.6458 +/- 0.0536,-,0.0107 +/- 0.0001,0.5836 +/- 0.0010,0.6787 +/- 0.0341,-,0.7117 +/- 0.0146,-
popt,0.5441 +/- 0.0041,0.5144 +/- 0.0097,0.0243 +/- 0.0152,0.5809 +/- 0.0342,-,0.0117 +/- 0.0002,0.5268 +/- 0.0142,0.5119 +/- 0.0159,-,0.5746 +/- 0.0075,-



=== Sig10 FM — Subject 8 (lag=0, test, mean +/- fold_std) ===


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5699 +/- 0.0021,0.4964 +/- 0.0087,0.0145 +/- 0.0046,0.7039 +/- 0.0441,-,0.0143 +/- 0.0000,0.5377 +/- 0.0039,0.6208 +/- 0.0846,-,0.6120 +/- 0.0011,-
diver,0.5018 +/- 0.0127,0.4929 +/- 0.0165,0.0056 +/- 0.0263,0.5362 +/- 0.0074,-,0.0109 +/- 0.0002,0.4883 +/- 0.0176,0.5663 +/- 0.0040,-,0.5820 +/- 0.0101,-
popt,0.5094 +/- 0.0064,0.4894 +/- 0.0007,0.0185 +/- 0.0060,0.6245 +/- 0.0382,-,0.0118 +/- 0.0002,0.5279 +/- 0.0150,0.5576 +/- 0.0406,-,0.5926 +/- 0.0044,-



=== Sig10 FM — Subject 9 (lag=0, test, mean +/- fold_std) ===


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5471 +/- 0.0035,0.5194 +/- 0.0063,0.0460 +/- 0.0090,0.6429 +/- 0.0129,-,0.0179 +/- 0.0001,0.5305 +/- 0.0082,0.5732 +/- 0.0553,0.0936 +/- 0.0136,0.6030 +/- 0.0086,-
diver,0.6068 +/- 0.0360,0.5629 +/- 0.0114,0.0939 +/- 0.0017,0.6806 +/- 0.0377,-,0.0107 +/- 0.0002,0.6094 +/- 0.0134,0.6210 +/- 0.0015,-,0.6999 +/- 0.0008,-
popt,0.5058 +/- 0.0081,0.5308 +/- 0.0178,0.0151 +/- 0.0111,0.4901 +/- 0.0466,-,0.0116 +/- 0.0002,0.5205 +/- 0.0179,0.5359 +/- 0.0028,-,0.5548 +/- 0.0088,-


## Save CSVs

In [105]:
# ── FM: Supersubject ──
mat_super_mean.to_csv(OUTPUT_DIR / 'fm_supersubject_mean.csv')
mat_super_std.to_csv(OUTPUT_DIR / 'fm_supersubject_fold_std.csv')

# ── FM: Subject_full ──
mean_subj_full.to_csv(OUTPUT_DIR / 'fm_subject_full_mean.csv')
std_subj_full.to_csv(OUTPUT_DIR / 'fm_subject_full_subject_std.csv')
for subj in SUBJECTS:
    per_subj_full[subj].to_csv(OUTPUT_DIR / f'fm_subject_full_sub{subj}_mean.csv')
    per_subj_full_std[subj].to_csv(OUTPUT_DIR / f'fm_subject_full_sub{subj}_fold_std.csv')

# ── FM: Sig10 ──
mean_sig10.to_csv(OUTPUT_DIR / 'fm_sig10_mean.csv')
std_sig10.to_csv(OUTPUT_DIR / 'fm_sig10_subject_std.csv')
for subj in SUBJECTS:
    per_subj_sig10[subj].to_csv(OUTPUT_DIR / f'fm_sig10_sub{subj}_mean.csv')
    per_subj_sig10_std[subj].to_csv(OUTPUT_DIR / f'fm_sig10_sub{subj}_fold_std.csv')

# ── FM: Persubject_concat ──
mat_psconcat_mean.to_csv(OUTPUT_DIR / 'fm_persubject_concat_mean.csv')
mat_psconcat_std.to_csv(OUTPUT_DIR / 'fm_persubject_concat_fold_std.csv')

# ── Baseline: baseline-results/ (all channels) ──
mat_bl_super_mean.to_csv(OUTPUT_DIR / 'baseline_allchan_mean.csv')
mat_bl_super_std.to_csv(OUTPUT_DIR / 'baseline_allchan_fold_std.csv')

# ── Baseline: sig10 ──
if bl_sig10_names:
    mean_bl_sig10.to_csv(OUTPUT_DIR / 'baseline_sig10_mean.csv')
    std_bl_sig10.to_csv(OUTPUT_DIR / 'baseline_sig10_subject_std.csv')
    for subj in SUBJECTS:
        per_subj_bl_sig10[subj].to_csv(OUTPUT_DIR / f'baseline_sig10_sub{subj}_mean.csv')
        per_subj_bl_sig10_std[subj].to_csv(OUTPUT_DIR / f'baseline_sig10_sub{subj}_fold_std.csv')

# ── Long-form CSV combining everything ──
rows = []

# FM — supersubject & persubject_concat
for cond in ['supersubject', 'persubject_concat']:
    for model in FOUNDATION_MODELS:
        for task in ALL_TASKS:
            m, s = fm[cond].get(model, {}).get(task, (np.nan, np.nan))
            rows.append({
                'type': 'FM', 'condition': cond, 'model': model, 'task': task,
                'subject': 'all', 'primary_metric': PRIMARY_METRIC[task],
                'value': m, 'fold_std': s
            })

# FM — subject_full & sig10
for cond in ['subject_full', 'sig10']:
    for subj in SUBJECTS:
        for model in FOUNDATION_MODELS:
            for task in ALL_TASKS:
                m, s = fm[cond][subj].get(model, {}).get(task, (np.nan, np.nan))
                rows.append({
                    'type': 'FM', 'condition': cond, 'model': model, 'task': task,
                    'subject': subj, 'primary_metric': PRIMARY_METRIC[task],
                    'value': m, 'fold_std': s
                })

# Baselines — baseline-results/ (all channels)
for bl_name in sorted(baseline_super.keys()):
    for task in ALL_TASKS:
        m, s = baseline_super[bl_name].get(task, (np.nan, np.nan))
        nf = baseline_meta.get((bl_name, task), None)
        rows.append({
            'type': 'baseline', 'condition': 'allchan', 'model': bl_name, 'task': task,
            'subject': 'all', 'primary_metric': PRIMARY_METRIC[task],
            'value': m, 'fold_std': s, 'n_folds': nf
        })

# Baselines — sig10
for subj in SUBJECTS:
    for bl_name in sorted(baseline_sig10[subj].keys()):
        for task in ALL_TASKS:
            m, s = baseline_sig10[subj].get(bl_name, {}).get(task, (np.nan, np.nan))
            nf = baseline_meta.get((bl_name, task), None)
            rows.append({
                'type': 'baseline', 'condition': 'sig10', 'model': bl_name, 'task': task,
                'subject': subj, 'primary_metric': PRIMARY_METRIC[task],
                'value': m, 'fold_std': s, 'n_folds': nf
            })

long_df = pd.DataFrame(rows)
long_df.to_csv(OUTPUT_DIR / 'all_conditions_long.csv', index=False)

# Print saved files
print('Saved files:')
for f in sorted(OUTPUT_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024 if f.is_file() else 0
    print(f'  {f.name:45s}  {size_kb:6.1f} KB')

Saved files:
  all_conditions_long.csv                          79.1 KB
  baseline_allchan_fold_std.csv                     0.7 KB
  baseline_allchan_mean.csv                         0.7 KB
  baseline_sig10_mean.csv                           0.4 KB
  baseline_sig10_sub1_fold_std.csv                  0.4 KB
  baseline_sig10_sub1_mean.csv                      0.4 KB
  baseline_sig10_sub2_fold_std.csv                  0.4 KB
  baseline_sig10_sub2_mean.csv                      0.4 KB
  baseline_sig10_sub3_fold_std.csv                  0.4 KB
  baseline_sig10_sub3_mean.csv                      0.4 KB
  baseline_sig10_sub4_fold_std.csv                  0.4 KB
  baseline_sig10_sub4_mean.csv                      0.4 KB
  baseline_sig10_sub5_fold_std.csv                  0.4 KB
  baseline_sig10_sub5_mean.csv                      0.4 KB
  baseline_sig10_sub6_fold_std.csv                  0.4 KB
  baseline_sig10_sub6_mean.csv                      0.4 KB
  baseline_sig10_sub7_fold_std.csv         